In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, TrainerCallback
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from tqdm.auto import tqdm

In [4]:
df = pd.read_csv("final_combined_dataset.csv")

In [5]:
def format_example(row):
    text = f"""You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT):
<answer>a/b/c/d/idk</answer>
<reasoning>2-3 line explanation</reasoning>

<answer>{row['final_answer']}</answer>
<reasoning>{row['reasoning']}</reasoning>
"""
    return {"text": text}

In [6]:
dataset = Dataset.from_list([format_example(row) for _, row in df.iterrows()])

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
training_args = TrainingArguments(
    output_dir="./qwen_mcq_sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=10,
    save_steps=200,
    fp16=True,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none",
    disable_tqdm=False
)

In [8]:
class TQDMProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps)

    def on_step_end(self, args, state, control, **kwargs):
        self.pbar.update(1)

    def on_train_end(self, args, state, control, **kwargs):
        self.pbar.close()


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    max_seq_length=1024,
    args=training_args,
    callbacks=[TQDMProgressCallback()]
)

trainer.train()

trainer.model.save_pretrained("qwen_mcq_sft_model")
tokenizer.save_pretrained("qwen_mcq_sft_model")

Map:   0%|          | 0/3521 [00:00<?, ? examples/s]

/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


  0%|          | 0/1320 [00:00<?, ?it/s]

Step,Training Loss
10,2.833400
20,2.747000
30,2.690500
40,2.425200
50,2.237000
60,1.882700
70,1.624300
80,1.372200
90,1.124400
100,1.043100


/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [1]:
!gpustat

dgx01                     Fri Apr 17 03:59:55 2026  535.183.06
[0] NVIDIA A100-SXM4-80GB | 30°C,  ?? % |     2 / 81920 MB |
[1] NVIDIA A100-SXM4-80GB | 29°C,  ?? % |    87 / 81920 MB |
[2] NVIDIA A100-SXM4-80GB | 30°C,  ?? % |    87 / 81920 MB |
[3] NVIDIA A100-SXM4-80GB | 31°C,  ?? % |    87 / 81920 MB |
[4] NVIDIA A100-SXM4-80GB | 34°C,  ?? % | 80700 / 81920 MB | sarmistha(44400M) sarmistha(20548M) sarmistha(12408M) sarmistha(3310M)
[5] NVIDIA A100-SXM4-80GB | 33°C,  ?? % | 69059 / 81920 MB | sarmistha(58402M) sarmistha(10634M)
[6] NVIDIA A100-SXM4-80GB | 35°C,  ?? % | 81036 / 81920 MB | sarmistha(36150M) sarmistha(27710M) sarmistha(17150M)
[7] NVIDIA A100-SXM4-80GB | 55°C,  ?? % | 14251 / 81920 MB | sarmistha(14112M)
